## 이커머스 리뷰 분석 에이전트

##### CrewAI와 상품 리뷰 CSV 데이터를 이용하여 리뷰를 감정·이슈로 분류하고, 트렌드를 분석해 제품/마케팅팀이 바로 쓸 수 있는 리포트를 생성하는 실습입니다.

##### 학습 목적으로만 사용 바랍니다.

In [ ]:
# 현재 노트북 커널에 CrewAI, 도구 패키지, pandas, 환경변수 로더를 설치합니다.
# 설치 후에는 Jupyter 커널을 한 번 재시작하는 것이 안전합니다.
%pip install -U "crewai[openai,tools]>=1.15,<2.0" "python-dotenv>=1.0" "pandas>=2.0" "pydantic>=2.0" "ipywidgets>=8.0"


In [ ]:
# ==================================================
# 환경변수 로드 및 라이브러리 불러오기
# ==================================================
from dotenv import load_dotenv
load_dotenv(override=True)

import os
from pathlib import Path
from typing import List, Literal

import pandas as pd
from pydantic import BaseModel
from IPython.display import Markdown, display

from crewai import Agent, Crew, LLM, Process, Task

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY를 찾을 수 없습니다. "
        "노트북과 같은 폴더의 .env 파일에 API Key를 설정하세요."
    )

model_name = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini")

llm = LLM(
    model=model_name,
    api_key=api_key,
    temperature=0.2,
)

print(f"사용 모델: {model_name}")


## `.env` 파일 예시

노트북과 같은 폴더에 `.env` 파일을 만들고 다음 값을 입력합니다.

```dotenv
OPENAI_API_KEY=sk-proj-실제_API_KEY
OPENAI_MODEL_NAME=openai/gpt-4o-mini
```


In [ ]:
# ==================================================
# 이커머스 리뷰 CSV 불러오기
# ==================================================
csv_path = Path("./ecommerce_review_data.csv")

if not csv_path.exists():
    raise FileNotFoundError(
        f"CSV 파일을 찾을 수 없습니다: {csv_path.resolve()}\n"
        "ecommerce_review_data.csv를 현재 노트북과 같은 폴더에 배치하세요."
    )

df = pd.read_csv(csv_path)
display(df.head())
print(f"총 리뷰 수: {len(df)}건 / 기간: {df['날짜(시간)'].min()} ~ {df['날짜(시간)'].max()}")


### 1단계 — 리뷰 분류 에이전트

리뷰 원문은 자유 텍스트라 감정·이슈를 판단하는 데 코드가 아니라 LLM의 해석이 필요합니다. `output_pydantic`으로 출력 스키마를 강제해서 결과를 그대로 DataFrame으로 쓸 수 있게 합니다.

API 비용과 정확도를 고려해 40건씩 배치로 나눠 분류합니다.

In [ ]:
# ==================================================
# 분류 결과 스키마 + 분류 에이전트/Task/Crew 정의
# ==================================================
class ReviewLabel(BaseModel):
    리뷰ID: str
    감정: Literal["긍정", "중립", "부정"]
    이슈카테고리: Literal["배송", "품질", "가격", "포장", "CS응대", "사이즈핏", "맛/풍미", "기타"]

class ReviewLabelBatch(BaseModel):
    labels: List[ReviewLabel]

review_classifier = Agent(
    role="리뷰 분류가",
    goal="리뷰 원문 {reviews}을 읽고 각 리뷰의 감정과 이슈카테고리를 정확히 태깅한다.",
    backstory=(
        "당신은 이커머스 상품 리뷰를 읽고 감정(긍정/중립/부정)과 이슈카테고리를 "
        "분류하는 전문가입니다. 리뷰ID는 절대 새로 만들지 않고 주어진 것만 사용하며, "
        "모든 리뷰를 빠짐없이 분류합니다."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

classification_task = Task(
    description=(
        "아래 리뷰들을 한 건도 빠짐없이 분류하세요. 형식은 '리뷰ID: 리뷰텍스트'입니다.\n\n"
        "{reviews}\n\n"
        "각 리뷰에 대해 감정(긍정/중립/부정)과 이슈카테고리 "
        "(배송/품질/가격/포장/CS응대/사이즈핏/맛·풍미/기타) 중 하나를 정확히 태깅하세요."
    ),
    expected_output="입력된 모든 리뷰ID에 대한 감정·이슈카테고리 라벨 목록",
    agent=review_classifier,
    output_pydantic=ReviewLabelBatch,
)

classification_crew = Crew(
    agents=[review_classifier],
    tasks=[classification_task],
    process=Process.sequential,
    verbose=True,
)


In [ ]:
# ==================================================
# 배치 분류 실행 (API 사용량 발생)
# ==================================================
BATCH_SIZE = 40
all_labels = []

for start in range(0, len(df), BATCH_SIZE):
    batch = df.iloc[start:start + BATCH_SIZE]
    review_lines = "\n".join(f"{r['리뷰ID']}: {r['리뷰원문']}" for _, r in batch.iterrows())
    result = await classification_crew.kickoff_async(inputs={"reviews": review_lines})
    all_labels.extend(result.pydantic.labels)
    print(f"{start + len(batch)}/{len(df)}건 분류 완료")

labels_df = pd.DataFrame([l.model_dump() for l in all_labels])
labeled_df = df.merge(labels_df, on="리뷰ID", how="left")
display(labeled_df.head())


### 2단계 — 집계 (코드가 계산, 에이전트 아님)

월별 감정비율, 상품×이슈별 전월대비 증감, 대표 부정 리뷰 샘플은 정답이 정해진 계산이라 pandas로 직접 처리합니다. 에이전트는 이 결과를 해석만 합니다.

In [ ]:
# ==================================================
# 월별 집계 / 이슈 급증 탐지 / 대표 부정 리뷰 샘플
# ==================================================
labeled_df["월"] = labeled_df["날짜(시간)"].str[:7]

monthly_sentiment = (
    labeled_df.groupby("월")["감정"]
    .value_counts(normalize=True).unstack(fill_value=0).round(3)
)

issue_counts = (
    labeled_df.groupby(["월", "상품", "이슈카테고리"]).size()
    .reset_index(name="건수")
)
issue_counts["전월건수"] = issue_counts.groupby(["상품", "이슈카테고리"])["건수"].shift(1).fillna(0)
issue_counts["증감"] = issue_counts["건수"] - issue_counts["전월건수"]
spikes = issue_counts[issue_counts["증감"] >= 4].sort_values("증감", ascending=False)

sample_negative = (
    labeled_df[labeled_df["감정"] == "부정"]
    .sort_values("날짜(시간)")
    .groupby(["상품", "이슈카테고리"]).tail(2)
    [["리뷰ID", "월", "상품", "이슈카테고리", "리뷰원문"]]
)

display(monthly_sentiment)
display(spikes.head(10))
display(sample_negative.head(10))


### 3단계 — 트렌드 분석·리포트 에이전트

집계된 통계와 대표 리뷰를 받아 해석하고, 제품/마케팅팀이 바로 실행할 수 있는 리포트로 종합합니다.

In [ ]:
# ==================================================
# Agent 정의
# ==================================================
trend_analyst = Agent(
    role="리뷰 트렌드 분석가",
    goal=(
        "사전에 계산된 월별 감정 분포 {monthly_sentiment}와 "
        "이슈 급증 데이터 {spikes}를 바탕으로 어떤 상품에서 어떤 문제가 커지고 있는지 해석한다."
    ),
    backstory=(
        "당신은 이커머스 리뷰 데이터를 해석하는 트렌드 분석가입니다. "
        "숫자는 이미 코드로 계산되어 주어지며, 당신은 그 숫자가 의미하는 바를 "
        "설명할 뿐 계산되지 않은 수치를 새로 만들어내지 않습니다."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

review_reporter = Agent(
    role="리뷰 인사이트 리포터",
    goal=(
        "{request}를 바탕으로 트렌드 분석 결과와 대표 부정 리뷰 {sample_negative}를 종합해 "
        "제품/마케팅팀이 바로 실행할 수 있는 리포트를 작성한다."
    ),
    backstory=(
        "당신은 리뷰 분석 결과를 제품/마케팅팀이 이해하기 쉬운 실행 리포트로 정리하는 "
        "전문가입니다. 대표 리뷰를 인용하며 우선순위와 개선 제안을 제시합니다."
    ),
    llm=llm,
    allow_delegation=False,
    verbose=True,
)


In [ ]:
# ==================================================
# Task 정의 및 Crew 구성
# ==================================================
trend_task = Task(
    description=(
        "다음 데이터를 바탕으로 상품별·이슈별 추세를 분석하세요.\n"
        "월별 감정분포: {monthly_sentiment}\n"
        "이슈 급증 목록: {spikes}\n"
        "어떤 상품에서 어떤 이슈가 급증했는지, 계절적 요인인지 이상 신호(장애/불량 등)인지 "
        "구분해서 설명하세요. 주어진 수치 외의 값을 지어내지 마세요."
    ),
    expected_output="상품별·이슈별 추세 요약과 급증 원인 추정을 포함한 한국어 분석",
    agent=trend_analyst,
)

report_task = Task(
    description=(
        "사용자 요청: {request}\n\n"
        "앞선 트렌드 분석과 대표 부정 리뷰 {sample_negative}를 참고하여 "
        "제품/마케팅팀이 바로 실행할 수 있는 리포트를 작성하세요. "
        "핵심 이슈 Top3, 대표 리뷰 인용, 개선 제안을 포함하세요."
    ),
    expected_output="핵심 이슈 Top3, 리뷰 인용, 개선 제안을 포함한 한국어 Markdown 리포트",
    agent=review_reporter,
    context=[trend_task],
)

insight_crew = Crew(
    agents=[trend_analyst, review_reporter],
    tasks=[trend_task, report_task],
    process=Process.sequential,
    verbose=True,
)


In [ ]:
# ==================================================
# Insight Crew 실행 (API 사용량 발생)
# ==================================================
request = "이번 분기 리뷰에서 가장 시급하게 대응해야 할 이슈를 정리해줘."

result = await insight_crew.kickoff_async(
    inputs={
        "request": request,
        "monthly_sentiment": monthly_sentiment.reset_index().to_dict(orient="records"),
        "spikes": spikes.to_dict(orient="records"),
        "sample_negative": sample_negative.to_dict(orient="records"),
    }
)

display(Markdown(result.raw))
